# Attention Map Visualization

Attention weights show how strongly each token attends to previous tokens while the model predicts the next character.
Visualizing them helps make the Transformer less abstract by showing which context positions each head uses.
This notebook loads a trained checkpoint, runs a short phrase through the model, and saves one heatmap per attention head.

## Imports

This cell imports the plotting and tensor libraries, then adds the project root to `sys.path` so the notebook runs both locally and in Colab after cloning the repository.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.py").exists() and (PROJECT_ROOT / "gpt-from-scratch").exists():
    PROJECT_ROOT = PROJECT_ROOT / "gpt-from-scratch"

sys.path.insert(0, str(PROJECT_ROOT))

## Path Configuration

This cell defines the checkpoint location, the output directory for generated attention maps, and the input text used for visualization.

In [ ]:
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "best_model.pt"
ASSETS_DIR = PROJECT_ROOT / "assets"
ASSETS_DIR.mkdir(exist_ok=True)

INPUT_TEXT = "To be or not to be"

## Load Model and Tokenizer

This cell loads the saved checkpoint, rebuilds the tokenizer vocabulary, reconstructs the GPT model, loads the trained weights, and switches the model to evaluation mode.

In [ ]:
from config import config
from model.gpt import build_model
from utils.tokenizer import CharacterTokenizer

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}. Run python train.py first.")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=config.device)
tokenizer = CharacterTokenizer(checkpoint["vocab"])

model = build_model(checkpoint["vocab_size"], checkpoint.get("config"))
model.load_state_dict(checkpoint["model_state"])
model.eval()

print(f"Loaded model with vocab_size={checkpoint['vocab_size']} on device={config.device}")

## Capture Attention Weights with a Forward Hook

A forward hook is a callback that runs automatically when a module performs its forward pass. Here it watches the first Transformer's attention module and stores the attention weights produced after softmax and before dropout.

In [ ]:
captured_weights = []


def attention_hook(module, inputs, output):
    captured_weights.append(module.last_attention_weights.detach().cpu())


hook_handle = model.blocks[0].attn.register_forward_hook(attention_hook)

## Forward Pass

This cell encodes the input text, runs a no-gradient forward pass, removes the hook, and prints the extracted character tokens.

In [ ]:
tokens = tokenizer.encode(INPUT_TEXT)
idx = torch.tensor([tokens], dtype=torch.long, device=config.device)

with torch.no_grad():
    logits, _ = model(idx)

hook_handle.remove()

token_chars = [tokenizer.decode([token]) for token in tokens]
print("Extracted tokens:", token_chars)

## Plot Attention Heatmaps

This cell creates one heatmap per head from layer 0. The x-axis shows key tokens, the y-axis shows query tokens, and the color indicates attention weight from 0 to 1.

In [ ]:
if not captured_weights:
    raise RuntimeError("No attention weights were captured. Re-run the hook and forward pass cells.")

attention_weights = captured_weights[0][0].numpy()
num_heads, sequence_length, _ = attention_weights.shape

for i in range(num_heads):
    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(attention_weights[i], cmap="viridis", vmin=0, vmax=1)

    ax.set_xticks(np.arange(sequence_length))
    ax.set_yticks(np.arange(sequence_length))
    ax.set_xticklabels(token_chars, rotation=90)
    ax.set_yticklabels(token_chars)

    ax.set_xlabel("Key tokens")
    ax.set_ylabel("Query tokens")
    ax.set_title(f"Layer 0 — Head {i}")

    fig.colorbar(image, ax=ax)
    fig.tight_layout()

    output_path = ASSETS_DIR / f"attention_layer0_head{i}.png"
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()

## How to Interpret the Maps

Each cell `(i, j)` shows how much query token `i` attends to key token `j`.
The causal mask guarantees that the upper triangular half is always zero because tokens cannot look into the future.
Different heads can specialize in different patterns, such as nearby positions, repeated characters, syntax-like structure, or line breaks.
As validation loss gets lower, attention maps usually become more structured and concentrated instead of uniformly diffuse.

## Next Steps

- Change `INPUT_TEXT` to another phrase and compare the attention patterns.
- Visualize deeper layers such as `blocks[1]`, `blocks[2]`, and beyond.
- Compare attention maps from the small 500-iteration model against the full 5000-iteration model.